In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch evaluate datasets accelerate peft')
    os.system('pip uninstall -y torchvision')
    print("Setup complete!")


In [2]:
# NOTE: Ensure you have `transformers`, `torch`, `evaluate`, and `accelerate` installed.
finetune_dir = 'datasets/finetuning'
output_model_dir = 'models/finetuned/xlm-roberta-base-langid'
model_name = "papluca/xlm-roberta-base-language-detection"
batch_size = 4
learning_rate = 2e-5
num_epochs = 10


In [3]:
import os
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

# Our target 10 languages
TARGET_LANGUAGES = {
    "eng": "en",  # English
    "hin": "hi",  # Hindi
    "arb": "ar",  # Arabic
    "fra": "fr",  # French
    "deu": "de",  # German
    # These below might not be in the original model, or we map them:
    "ben": "bn",
    "tam": "ta",
    "sin": "si",
    "san": "sa",
    "pli": "pi",
}

print("Loading original model configuration...")
config = AutoConfig.from_pretrained(model_name)

# Add our new labels to the config if they don't exist
added_labels = []
for old_code, new_code in TARGET_LANGUAGES.items():
    if new_code not in config.label2id:
        idx = len(config.label2id)
        config.label2id[new_code] = idx
        config.id2label[idx] = new_code
        added_labels.append(new_code)

print(f"Added {len(added_labels)} new labels: {added_labels}")
print(f"Total labels in model: {len(config.label2id)}")

def load_data(jsonl_path):
    records = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            # Map our 3-letter codes to the 2-letter codes expected by the model
            mapped = TARGET_LANGUAGES.get(rec['label'], rec['label'])
            if mapped in config.label2id:
                records.append({
                    "text": rec["text"],
                    "label": config.label2id[mapped]
                })
    return pd.DataFrame(records)

print("\nLoading datasets...")
train_df = load_data(os.path.join(finetune_dir, "train.jsonl"))
val_mixed_df = load_data(os.path.join(finetune_dir, "val_mixed.jsonl"))

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_mixed_df)}")


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading original model configuration...
Added 5 new labels: ['bn', 'ta', 'si', 'sa', 'pi']
Total labels in model: 25

Loading datasets...
Train size: 60285
Validation size: 10486


In [4]:
from datasets import Dataset as HFDataset

print("Tokenizing datasets...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = HFDataset.from_pandas(train_df)
val_dataset = HFDataset.from_pandas(val_mixed_df)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Format for PyTorch
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_val = tokenized_val.remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


Tokenizing datasets...


Map: 100%|██████████| 10486/10486 [00:00<00:00, 13097.97 examples/s]


In [5]:
print("Loading model and expanding classification head...")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

old_out_features = model.classifier.out_proj.out_features
new_out_features = len(config.label2id)

if new_out_features > old_out_features:
    print(f"Expanding classification head from {old_out_features} to {new_out_features} classes...")
    new_out_proj = torch.nn.Linear(model.classifier.out_proj.in_features, new_out_features)
    
    # Copy old weights
    new_out_proj.weight.data[:old_out_features] = model.classifier.out_proj.weight.data
    new_out_proj.bias.data[:old_out_features] = model.classifier.out_proj.bias.data
    
    # Initialize new weights safely
    torch.nn.init.xavier_uniform_(new_out_proj.weight.data[old_out_features:])
    torch.nn.init.zeros_(new_out_proj.bias.data[old_out_features:])
    
    model.classifier.out_proj = new_out_proj
    model.num_labels = new_out_features
    model.config = config


from peft import get_peft_model, LoraConfig, TaskType
import torch.distributed.tensor

print("Applying LoRA to freeze base weights and inject adapters...")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=256,
    lora_alpha=512,
    lora_dropout=0.1,
    # target query and value attention matrices
    target_modules=["query", "key", "value", "dense"], 
    # train the expanded classification head
    modules_to_save=["classifier"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready for finetuning.")


Loading model and expanding classification head...
Expanding classification head from 20 to 25 classes...
Applying LoRA to freeze base weights and inject adapters...
trainable params: 43,077,145 || all params: 321,140,018 || trainable%: 13.4138
Model ready for finetuning.


In [6]:
import evaluate
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# We use Micro F1 as requested by the user
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="micro")

training_args = TrainingArguments(
    output_dir=output_model_dir,
    eval_strategy="epoch",  # Evaluate every epoch
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=16 // batch_size,
    fp16=torch.cuda.is_available(),
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True, # Critical for Early Stopping
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none" # Disable wandb/tensorboard for simplicity
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if F1 drops for 2 consecutive epochs
)

print("Starting Fine-tuning...")
trainer.train()

print(f"Saving final model to {output_model_dir}...")
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print("Finetuning Complete!")


Starting Fine-tuning...


  1%|▏         | 500/37680 [02:29<3:05:06,  3.35it/s]

{'loss': 0.2261, 'grad_norm': 0.051749713718891144, 'learning_rate': 1.9736730360934185e-05, 'epoch': 0.13}


  3%|▎         | 1000/37680 [04:58<3:00:48,  3.38it/s]

{'loss': 0.038, 'grad_norm': 0.033466190099716187, 'learning_rate': 1.947186836518047e-05, 'epoch': 0.27}


...

{'loss': 0.0141, 'grad_norm': 0.03298680856823921, 'learning_rate': 1.204299363057325e-05, 'epoch': 3.98}


                                                         
 40%|████      | 15072/37680 [1:20:04<1:47:38,  3.50it/s]

{'eval_loss': 6.378268718719482, 'eval_f1': 0.664218958611482, 'eval_runtime': 67.7804, 'eval_samples_per_second': 154.706, 'eval_steps_per_second': 38.684, 'epoch': 4.0}


 40%|████      | 15072/37680 [1:20:06<2:00:09,  3.14it/s]


{'train_runtime': 4806.3182, 'train_samples_per_second': 125.429, 'train_steps_per_second': 7.84, 'train_loss': 0.023704174146720557, 'epoch': 4.0}
Saving final model to models/finetuned/xlm-roberta-base-langid...
Finetuning Complete!
